# KoELECTRA 이진 분류기 — 이어폰 vs 비이어폰

`title` 컬럼만 보고 이어폰(1) / 비이어폰(0) 을 분류하는 모델을 파인튜닝합니다.
- **학습셋**: `category4`가 채워진 884개
- **테스트셋**: `category4`가 비어 있는 116개

## Step 1. 데이터 준비

In [2]:
import re
import pandas as pd

df_raw = pd.read_csv("naver_무선이어폰.csv", encoding="utf-8-sig")

# HTML 태그 제거
df_raw["title_clean"] = df_raw["title"].apply(lambda x: re.sub("<.*?>", "", x))

# NaN을 빈 문자열로 통일
df_raw["category4"] = df_raw["category4"].fillna("")

train = df_raw[df_raw["category4"] != ""].copy()
test  = df_raw[df_raw["category4"] == ""].copy()

# 이어폰(1) vs 비이어폰(0) — 헤드폰/헤드셋은 0으로 처리
train["label"] = (train["category4"] == "블루투스이어폰/이어셋").astype(int)

print(f"학습셋: {len(train)}개  (이어폰 {train['label'].sum()}개 / 비이어폰 {(train['label']==0).sum()}개)")
print(f"테스트셋: {len(test)}개 (category4 미분류)")
train["label"].value_counts()

학습셋: 878개  (이어폰 553개 / 비이어폰 325개)
테스트셋: 122개 (category4 미분류)


,count
label,
1,553
0,325


## Step 2. 모델 & 토크나이저 로드

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "monologg/koelectra-base-v3-discriminator" # BERT 종류
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
print("모델 로드 완료:", model_name)

config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/61.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/263k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/452M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: monologg/koelectra-base-v3-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your do

모델 로드 완료: monologg/koelectra-base-v3-discriminator


## Step 3. Dataset 클래스

In [4]:
import torch
from torch.utils.data import Dataset

class ProductDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=64):
        self.encodings = tokenizer(
            texts, truncation=True, padding=True,
            max_length=max_len, return_tensors="pt"
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

## Step 4. 학습 (Trainer API)

In [5]:
import numpy as np
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

train_ds = ProductDataset(
    train["title_clean"].tolist(),
    train["label"].tolist(),
    tokenizer
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    learning_rate=3e-5,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

Step,Training Loss
50,0.571314
100,0.370808
150,0.298014
200,0.206655
250,0.139799


TrainOutput(global_step=275, training_loss=0.30752632834694604, metrics={'train_runtime': 44.8587, 'train_samples_per_second': 97.863, 'train_steps_per_second': 6.13, 'total_flos': 133102332907800.0, 'train_loss': 0.30752632834694604, 'epoch': 5.0})

## Step 5. 테스트셋 예측 및 결과 확인

In [6]:
test_ds = ProductDataset(test["title_clean"].tolist(), tokenizer=tokenizer)
preds = trainer.predict(test_ds)
pred_labels = np.argmax(preds.predictions, axis=1)

result = test[["title_clean"]].copy()
result["pred"] = pred_labels
result["pred_label"] = result["pred"].map({1: "이어폰 ✓", 0: "비이어폰 ✗"})

print(f"이어폰: {(pred_labels==1).sum()}개 / 비이어폰: {(pred_labels==0).sum()}개\n")
display(result[["title_clean", "pred_label"]])

이어폰: 106개 / 비이어폰: 16개



,title_clean,pred_label
1,NAVEE NV2O-WE1 오픈형 블루투스이어폰 귀안아픈 무선이어폰 등산 조깅 스포...,이어폰 ✓
2,BEV7 블루투스 이어폰 무선 헤드셋 이어셋 보조배터리 BEV7블루투스 이어폰,이어폰 ✓
3,통화 음악감상 휴대 무선 블루투스 이어폰 블랙,이어폰 ✓
4,아이리버 무선이어폰 오픈핏 귀걸이형 블루투스 IB-iX7 블랙,이어폰 ✓
5,엠지텍 통화용 2대 멀티포인트 퀄컴 APT-X코덱 무선 블루투스 이어폰 C-ALL ...,이어폰 ✓
...,...,...
979,OMIIYA 터치스크린 블루투스 이어폰 ANC 노이즈캔슬링 커널형 무선이어폰 블랙...,이어폰 ✓
980,아이리버 프리미엄 사운드 TWS 커널형 무선 V5.3 블루투스 이어폰 IBE-G03...,이어폰 ✓
982,Dell Pro Plus ANC노이즈캔슬링 무선 블루투스 이어버드 블랙 EB525,이어폰 ✓
983,아이리버 노이즈캔슬링 블루투스 무선이어폰 ITW-ANC4 ITW-ANC4 블랙 ...,이어폰 ✓
